# 2026–2027 Deep Learning Final Coursework — Part A
## Explore the tactile dataset

This notebook helps you download, check and understand the released data before designing your own models. **Part A includes both supplied tasks:** shape classification (called “Task A” in the original dataset guide) and force estimation (“Task B”).

You will inspect 2,000 shape images and 2,000 force images, read their labels, view the ten shape categories and compare the press/shear images belonging to one contact. No training or assessment solution is provided.

**Run the cells from top to bottom.** A CPU is sufficient for this exploration. The repository's [coursework guide](../docs/COURSEWORK_GUIDE.md) describes the required splits, evaluation and submission interface. The data is a coursework subset of [TacVerse](https://doi.org/10.1002/adrr.70169), supplied with the original dataset README.


## 1. Choose your environment

**Google Colab:** open this notebook in Colab and run it. The setup cell clones the coursework repository if necessary and extracts the dataset under `/content`. Files stored in a temporary Colab runtime may need to be downloaded again after it ends.

**Local Jupyter / VS Code:** clone or download this repository, open this notebook and choose your Python environment. The dataset is extracted inside the repository. If an import fails, install the inspection dependencies in that environment:

```bash
python -m pip install numpy pandas pillow matplotlib
```

The download is pinned to release `v1.0.0` (37.44 MB). The helper checks the complete archive's SHA-256 before extraction. It preserves existing dataset folders.


In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

REPO_NAME = "2026-2027-Deep-Learning-Final-Coursework-Part-A"
REPO_URL = f"https://github.com/dianzdd123/{REPO_NAME}.git"

# Find an existing checkout, whether this notebook starts in the root or notebooks/.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / REPO_NAME]
REPO = next((p for p in candidates if (p / "scripts/download_dataset.py").is_file()), None)
if REPO is None:
    REPO = Path.cwd() / REPO_NAME
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

# Colab uses /content; a local session uses the checkout directory.
DATA_PARENT = Path("/content") if Path("/content").is_dir() else REPO
ROOT = DATA_PARENT / "Tactile_DL_2027"
if not ROOT.exists():
    subprocess.run(
        [sys.executable, str(REPO / "scripts/download_dataset.py"), "--output-dir", str(DATA_PARENT)],
        check=True,
    )

# Re-check an existing extraction as well as a fresh download.
subprocess.run(
    [sys.executable, str(REPO / "scripts/verify_dataset.py"), "--data", str(ROOT)],
    check=True,
)
print("Dataset location:", ROOT)


### Already have the ZIP?

You can set up a local copy without another download by running the following command in a terminal, then returning to the notebook:

```bash
python scripts/download_dataset.py --archive /path/to/Tactile_DL_2027.zip
```

The helper accepts the uploaded filename `Tactile_DL_2027 (1).zip` too; quote paths containing spaces. The contents and checksum, rather than the filename, identify the release.


## 2. Read the labels

Each `filename` is relative to **its task folder**, not to the repository. For example, join a shape filename with `ROOT / "shape_classification"`.

The shape task has one integer target, `class_id`. The force task has three targets in **newtons**: `fx`, `fy` and `fz`. `contact_id` links a normal press to its subsequent shear image. `is_shear` is useful for evaluation and exploration; it is not an input available to the hidden-test model.


In [ ]:
shape_root = ROOT / "shape_classification"
force_root = ROOT / "force_regression"
shape = pd.read_csv(shape_root / "labels.csv")
classes = pd.read_csv(shape_root / "classes.csv").sort_values("class_id")
force = pd.read_csv(force_root / "labels.csv")

print(f"Shape images: {len(shape):,}")
print(f"Force images: {len(force):,}; physical contacts: {force.contact_id.nunique():,}")
display(shape.head())
display(force.head())


## 3. Check the shape labels and view examples

Use the provided class IDs throughout your work. Do not recreate a different mapping by sorting or renaming the labels. The released set contains **200 images per class**.

The examples below are the first labelled image for each class; they are illustrative, not a ranking of representative or difficult examples. These are tactile camera images of contact, rather than ordinary photographs of the objects.


In [ ]:
class_counts = shape.groupby(["class_id", "class_name"]).size().rename("images").reset_index()
display(class_counts)
assert len(classes) == 10
assert class_counts.images.eq(200).all()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, label in zip(axes.flat, classes.itertuples(index=False)):
    row = shape.loc[shape.class_id.eq(label.class_id)].iloc[0]
    with Image.open(shape_root / row.filename) as image:
        ax.imshow(image)
    ax.set_title(f"{label.class_id}: {label.class_name}")
    ax.axis("off")
fig.suptitle("One released tactile image for each shape class")
plt.tight_layout()
plt.show()


In [ ]:
# Images are 320 pixels wide and 240 pixels high.
# NumPy reports array dimensions as (height, width, colour channels).
example_path = shape_root / shape.iloc[0].filename
with Image.open(example_path) as image:
    pixels = np.asarray(image.convert("RGB"))
    print("Pillow size (width, height):", image.size)
print("NumPy shape (height, width, channels):", pixels.shape)
print("Stored pixel type:", pixels.dtype)


## 4. Inspect paired force observations

Each `contact_id` has two images: `is_shear = 0` for the normal press, and `is_shear = 1` after a lateral displacement. **Keep both images in the same split.** Separating the pair between training and testing would leak information about the same contact.

`fx` and `fy` are tangential components. `fz` is normal force; the supplied convention uses negative `fz` for pressing into the sensor. The hidden-test filenames are anonymous, so a model must estimate force from image content alone.


In [ ]:
contact = int(force.contact_id.min())
pair = force.loc[force.contact_id.eq(contact)].sort_values("is_shear")
assert len(pair) == 2 and set(pair.is_shear) == {0, 1}
display(pair)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, row in zip(axes, pair.itertuples(index=False)):
    with Image.open(force_root / row.filename) as image:
        ax.imshow(image)
    kind = "Normal press" if row.is_shear == 0 else "After shear"
    ax.set_title(f"{kind} — contact {row.contact_id}\nfx={row.fx:.3f}, fy={row.fy:.3f}, fz={row.fz:.3f} N")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Read the force distribution

These tables describe the released subset. They are **not preprocessing parameters**. After creating your split, calculate any means and standard deviations used for normalisation from the **training split only**, then apply those fixed values to validation and test data. Convert predictions back to newtons before reporting errors.

Compare the tangential spread for press and shear images: an overall score can hide weak shear estimation because half of the observations are normal presses. The coursework therefore requires separate press/shear results and a training-mean baseline.


In [ ]:
force_axes = ["fx", "fy", "fz"]
print("All released observations; standard deviation is sample SD (ddof=1):")
display(force[force_axes].describe().loc[["count", "mean", "std", "min", "max"]])

print("Press (0) versus shear (1), in newtons:")
display(force.groupby("is_shear")[force_axes].agg(["mean", "std"]))

# Every contact should contribute exactly one image of each kind.
pair_counts = force.groupby(["contact_id", "is_shear"]).size().unstack()
assert pair_counts.shape == (1000, 2)
assert pair_counts.eq(1).all().all()
print("Verified: 1,000 complete press/shear pairs.")


## 6. Plan your own experiment

The supplied dataset guide requires **60% training, 20% validation and 20% test**:

| Task | Split unit | Training | Validation | Test |
|---|---|---:|---:|---:|
| Shape classification | Images, stratified by class | 1,200 images | 400 images | 400 images |
| Force estimation | Whole `contact_id` groups | 600 contacts / 1,200 images | 200 contacts / 400 images | 200 contacts / 400 images |

Fix and record your random seed. For shape classification, stratification gives 120/40/40 images per class. For force estimation, verify that the contact-ID sets do not overlap. Use validation data for model selection and keep your test split for final evaluation.

Before training, consider the following:

- What contact details might disappear if you reduce image resolution?
- Which transformations preserve the task's labels? Rotating or flipping a force image may require a corresponding change to force-vector labels; the image-to-force coordinate mapping must be justified.
- What does a confusion matrix reveal that accuracy alone does not?
- Does a force model outperform the training-set mean on the shear subset, especially for `fx` and `fy`?

The archive includes `example_submission/predict.py`, its example weights and the script used to produce them. They demonstrate the required inference interface. **The supplied example weights were trained on all released images; they do not provide a valid coursework evaluation.** See the [coursework guide](../docs/COURSEWORK_GUIDE.md) before preparing your submission.
